<a href="https://colab.research.google.com/github/OlhaZahrebelna/certflow-rag-assistant/blob/main/src_retrieval_01_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup

In [1]:
!pip install -q sentence-transformers faiss-cpu

In [2]:
import json
from sentence_transformers import SentenceTransformer
from pathlib import Path
import faiss
import numpy as np

In [3]:
from pathlib import Path

repo_path = Path("/content/certflow-rag-assistant")

if not repo_path.exists():
    !git clone https://github.com/OlhaZahrebelna/certflow-rag-assistant.git
else:
    print("Repository already exists.")

Repository already exists.


In [4]:
%cd /content/certflow-rag-assistant

/content/certflow-rag-assistant


In [5]:
chunks_path = Path("data/raw/processed/chunks.json")

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded chunks: {len(chunks)}")

Loaded chunks: 81


### Load chunks

In [6]:
texts = [chunk["content"] for chunk in chunks]

print(f"Texts prepared: {len(texts)}")

Texts prepared: 81


### multi-qa embeddings

In [7]:
model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

(81, 384)


### FAISS

In [9]:
embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"Vectors in index: {index.ntotal}")

Vectors in index: 81


## semantic_search()

In [10]:
def semantic_search(query, k=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    scores, indices = index.search(
        query_embedding,
        k=k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "chunk_id": chunk["chunk_id"],
            "document": chunk["metadata"]["title"],
            "section": chunk["metadata"]["section"],
            "content": chunk["content"],
        })

    return results

### 20-query evaluation

In [11]:
semantic_search("What evidence is required before an account can be certified?")

[{'rank': 1,
  'score': 0.5981685519218445,
  'chunk_id': 'ACD-KB-001-chunk-001',
  'document': 'Account Data Certification Overview',
  'section': '1. Purpose',
  'content': 'Account Data Certification is the controlled process used by Atlas Data Services (ADS) to confirm that a company account record is complete, accurate, consistent, traceable, and suitable for operational and analytical use. The certified object is the **account record**, not a person or organization receiving a professional credential.\n\nCertification supports account onboarding, duplicate prevention, entity matching, reporting, segmentation, customer communication, and synchronization between HorizonCRM, MeridianMDM, and BeaconLake.'},
 {'rank': 2,
  'score': 0.5937610864639282,
  'chunk_id': 'ACD-KB-003-chunk-001',
  'document': 'End-to-End Account Certification Workflow',
  'section': '1. Intake',
  'content': 'Certification starts from a scheduled review, auto-created account review, stakeholder request, data

In [12]:
results = semantic_search(
    "What evidence is required before an account can be certified?"
)

for result in results:
    print(
        result["rank"],
        round(result["score"], 4),
        result["document"],
        "→",
        result["section"]
    )

1 0.5982 Account Data Certification Overview → 1. Purpose
2 0.5938 End-to-End Account Certification Workflow → 1. Intake
3 0.593 Account Certification Frequently Asked Questions → General
4 0.5573 End-to-End Account Certification Workflow → 9. Certification decision
5 0.5422 Source Hierarchy and Evidence Standard → 6. Evidence recording


In [13]:
def evaluate_section_hit_at_k(evaluation_dataset, k=5):
    hits = 0
    details = []

    for item in evaluation_dataset:
        results = semantic_search(item["query"], k=k)

        is_hit = False
        hit_rank = None

        for result in results:
            for expected in item["expected_sections"]:

                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    is_hit = True
                    hit_rank = result["rank"]
                    break

            if is_hit:
                break

        if is_hit:
            hits += 1

        details.append({
            "query": item["query"],
            "hit": is_hit,
            "rank": hit_rank,
        })

    return hits / len(evaluation_dataset), details

In [14]:
def evaluate_section_mrr(evaluation_dataset, k=5):
    reciprocal_ranks = []

    for item in evaluation_dataset:
        results = semantic_search(item["query"], k=k)

        rr = 0

        for result in results:
            for expected in item["expected_sections"]:

                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    rr = 1 / result["rank"]
                    break

            if rr > 0:
                break

        reciprocal_ranks.append(rr)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [15]:
evaluation_dataset = [
    {
        "query": "What evidence is required before an account can be certified?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "6. Evidence recording"
            },
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "4. Gather evidence"
            }
        ]
    },
    {
        "query": "How should potential duplicate accounts be handled?",
        "expected_sections": [
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "6. Duplicate screening"
            },
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "7. Duplicate outcomes"
            }
        ]
    },
    {
        "query": "When should a certification case be escalated?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "6. Escalation levels"
            }
        ]
    },
    {
        "query": "Who is responsible for performing the quality assurance review?",
        "expected_sections": [
            {
                "document": "Roles and Responsibilities",
                "section": "4. Quality Assurance Reviewer"
            }
        ]
    },
    {
        "query": "What validation rules apply to account fields?",
        "expected_sections": [
            {
                "document": "Account Fields and Validation Rules",
                "section": "2. Core field catalog"
            }
        ]
    },
    {
        "query": "What should an analyst do when two sources contain conflicting information?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "5. Conflicting sources"
            }
        ]
    },
    {
        "query": "Which sources should be preferred when verifying account data?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "2. Primary sources"
            },
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "3. Secondary sources"
            }
        ]
    },
    {
        "query": "What should be recorded when a certification change is made?",
        "expected_sections": [
            {
                "document": "Request Types and Change Management",
                "section": "4. Required audit record"
            },
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "6. Record the change"
            }
        ]
    },
    {
        "query": "What information is required when submitting a new certification request?",
        "expected_sections": [
            {
                "document": "Request Types and Change Management",
                "section": "2. Intake requirements"
            },
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "1. Intake"
            }
        ]
    },
    {
        "query": "How do we verify that we are working with the correct company entity?",
        "expected_sections": [
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "2. Identify the correct entity"
            }
        ]
    },
    {
        "query": "What checks are needed before setting an account to Verified?",
        "expected_sections": [
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "9. Certification decision"
            }
        ]
    },
    {
        "query": "When is quality assurance mandatory?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "2. Mandatory QA triggers"
            }
        ]
    },
    {
        "query": "What should be checked during a QA review?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "3. QA checklist"
            }
        ]
    },
    {
        "query": "What happens when an account does not pass quality review?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "4. QA outcomes"
            }
        ]
    },
    {
        "query": "How should missing address information be handled?",
        "expected_sections": [
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "8. Missing address information"
            }
        ]
    },
    {
        "query": "How should an account address be normalized?",
        "expected_sections": [
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "5. Normalization"
            }
        ]
    },
    {
        "query": "What rules apply when the legal name of an account is updated?",
        "expected_sections": [
            {
                "document": "Account Fields and Validation Rules",
                "section": "3. Legal Name"
            }
        ]
    },
    {
        "query": "How should website and primary domain values be validated?",
        "expected_sections": [
            {
                "document": "Account Fields and Validation Rules",
                "section": "5. Website and Primary Domain"
            }
        ]
    },
    {
        "query": "What should happen when a relevant source is unavailable?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "8. Unavailable sources"
            }
        ]
    },
    {
        "query": "When does a certified account need to be reviewed again?",
        "expected_sections": [
            {
                "document": "Account Data Certification Overview",
                "section": "6. Certification validity"
            }
        ]
    }
]

In [16]:
section_hit_rate, section_details = evaluate_section_hit_at_k(
    evaluation_dataset,
    k=5
)

section_mrr = evaluate_section_mrr(
    evaluation_dataset,
    k=5
)

print(f"Section Hit@5: {section_hit_rate:.2f}")
print(f"Section MRR@5: {section_mrr:.3f}")

Section Hit@5: 0.85
Section MRR@5: 0.618


In [19]:
for item in section_details:
    if not item["hit"]:
        print("\nMISS")
        print("Query:", item["query"])
        print("Rank:", item["rank"])


MISS
Query: When should a certification case be escalated?
Rank: None

MISS
Query: What should be recorded when a certification change is made?
Rank: None

MISS
Query: What happens when an account does not pass quality review?
Rank: None


## BGE experiment

In [20]:
bge_model = SentenceTransformer("BAAI/bge-small-en-v1.5")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [21]:
bge_embeddings = bge_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

bge_embeddings = np.asarray(
    bge_embeddings,
    dtype="float32"
)

print("BGE embeddings shape:", bge_embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

BGE embeddings shape: (81, 384)


In [22]:
bge_dimension = bge_embeddings.shape[1]

bge_index = faiss.IndexFlatIP(bge_dimension)
bge_index.add(bge_embeddings)

print("Vectors in BGE index:", bge_index.ntotal)

Vectors in BGE index: 81


In [23]:
def bge_semantic_search(query, k=5):
    query_embedding = bge_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    scores, indices = bge_index.search(
        query_embedding,
        k=k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "chunk_id": chunk["chunk_id"],
            "document": chunk["metadata"]["title"],
            "section": chunk["metadata"]["section"],
            "content": chunk["content"],
        })

    return results

In [24]:
def evaluate_bge_hit_at_k(evaluation_dataset, k=5):
    hits = 0
    details = []

    for item in evaluation_dataset:
        results = bge_semantic_search(
            item["query"],
            k=k
        )

        is_hit = False
        hit_rank = None

        for result in results:
            for expected in item["expected_sections"]:
                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    is_hit = True
                    hit_rank = result["rank"]
                    break

            if is_hit:
                break

        if is_hit:
            hits += 1

        details.append({
            "query": item["query"],
            "hit": is_hit,
            "rank": hit_rank
        })

    return hits / len(evaluation_dataset), details

In [25]:
def evaluate_bge_mrr(evaluation_dataset, k=5):
    reciprocal_ranks = []

    for item in evaluation_dataset:
        results = bge_semantic_search(
            item["query"],
            k=k
        )

        rr = 0

        for result in results:
            for expected in item["expected_sections"]:
                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    rr = 1 / result["rank"]
                    break

            if rr > 0:
                break

        reciprocal_ranks.append(rr)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [26]:
bge_hit_rate, bge_details = evaluate_bge_hit_at_k(
    evaluation_dataset,
    k=5
)

bge_mrr = evaluate_bge_mrr(
    evaluation_dataset,
    k=5
)

print(f"BGE Section Hit@5: {bge_hit_rate:.2f}")
print(f"BGE Section MRR@5: {bge_mrr:.3f}")

BGE Section Hit@5: 0.75
BGE Section MRR@5: 0.499


In [27]:
for item in bge_details:
    if not item["hit"]:
        print("\nMISS")
        print("Query:", item["query"])
        print("Rank:", item["rank"])


MISS
Query: What evidence is required before an account can be certified?
Rank: None

MISS
Query: When should a certification case be escalated?
Rank: None

MISS
Query: Which sources should be preferred when verifying account data?
Rank: None

MISS
Query: What should be recorded when a certification change is made?
Rank: None

MISS
Query: What happens when an account does not pass quality review?
Rank: None


### Result

`multi-qa-MiniLM-L6-cos-v1` outperformed `BAAI/bge-small-en-v1.5`
on both retrieval coverage and ranking quality.

Therefore, `multi-qa-MiniLM-L6-cos-v1` is retained as the dense retrieval baseline
for the next retrieval experiments.

## Embedding Model Comparison

Two embedding models were evaluated on the same 20-query retrieval dataset.

| Model | Hit@5 | MRR@5 |
|---|---:|---:|
| multi-qa-MiniLM-L6-cos-v1 | 0.85 | 0.618 |
| BAAI/bge-small-en-v1.5 | 0.75 | 0.499 |

`multi-qa-MiniLM-L6-cos-v1` was selected as the dense retrieval baseline because it achieved better retrieval coverage and ranking quality.